<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">湖表与内部表关联</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">查询湖上的订单，与 Doris 客户表关联并核对导入结果。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

具备讲师预置的 Iceberg 环境后，你将直查十笔湖上订单、关联内部客户表，并对导入结果核对十行、金额 12220.60。请先按下方环境要求准备。

[讲义](course4_querying_external_data.md) · [课程入口](../README.md)


## 环境要求与状态

本实验需要讲师预置 Iceberg 服务，并通过 Doris External Catalog 提供订单表。表中应包含 [WWI 样本](../../datasets/wwi/sample.json) 中 orders 的六个字段与十行数据。

启动 Jupyter 前设置 DW_ICEBERG_ORDERS=catalog.database.table，填写讲师提供的实际表名。准备好环境后再执行；也可以先完成讲义与测验，继续 D05，之后补做本实验。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import os
from dw_course.runtime import identifier
source_parts = os.environ["DW_ICEBERG_ORDERS"].split(".")
if len(source_parts) != 3:
    raise ValueError("Expected catalog.database.table")
source = ".".join(identifier(part) for part in source_parts)
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. 直查湖表

Doris 通过 Catalog 找到 Iceberg 表，再读取它的数据文件。下一格检查表类型、查询计划与订单内容。预期为十笔订单、税前金额 12220.60；此时订单仍保存在湖表中。


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"12220.60")])
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1");


## 2. 与内部客户表关联

本步骤仅重建 customers_sample 与 orders_from_lake。先建立唯一键客户表，与湖上订单按 customer_id 关联：每笔订单应恰好找到一条客户记录。

再将六个订单字段导入 orders_from_lake，逐字段比较湖表与内部表。关联和导入完成后，仍应有十笔订单、税前金额 12220.60。


In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")])
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows())
lab.close()


## 完成与边界

能够通过 Catalog 查询 Iceberg 订单表；关联内部客户后订单行数和金额保持一致；导入后的内部表逐字段匹配湖表。


## 自己动手

对比直查湖表与查询 orders_from_lake 的执行计划，找出各自的扫描对象。说明 Iceberg 表元数据在查询中的作用，以及为什么客户维表需要按 customer_id 保持唯一。
